<a href="https://colab.research.google.com/github/Rachana826/Dental-Clinic-project/blob/main/Dental_Clinic_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq
from dataclasses import dataclass
from typing import List, Optional, Tuple

CLINIC_OPEN_HOUR = 9
CLINIC_CLOSE_HOUR = 17
MIN_DURATION = 1
MAX_DURATION = 8


@dataclass
class Appointment:

    patient: str
    duration: int
    priority: int = 5

    operatory_id: Optional[int] = None
    dentist_id: Optional[int] = None
    start: Optional[int] = None
    end: Optional[int] = None

    def __post_init__(self):
        if not (MIN_DURATION <= self.duration <= MAX_DURATION):
            raise ValueError(
                f"{self.patient}: duration must be between "
                f"{MIN_DURATION} and {MAX_DURATION} hours, got {self.duration}"
            )

    def clone(self) -> "Appointment":

        return Appointment(self.patient, self.duration, self.priority)


class ClinicScheduler:

    def __init__(self, num_operatories: int, num_dentists: int):
        if not (1 <= num_operatories <= 10):
            raise ValueError("Number of operatories must be between 1 and 10")
        if not (1 <= num_dentists <= 10):
            raise ValueError("Number of dentists must be between 1 and 10")

        self.num_operatories = num_operatories
        self.num_dentists = num_dentists

        self._op_heap: List[Tuple[int, int]] = [
            (CLINIC_OPEN_HOUR, i) for i in range(1, num_operatories + 1)
        ]
        self._dent_heap: List[Tuple[int, int]] = [
            (CLINIC_OPEN_HOUR, i) for i in range(1, num_dentists + 1)
        ]
        heapq.heapify(self._op_heap)
        heapq.heapify(self._dent_heap)

        self.schedule: List[Appointment] = []
        self.unscheduled: List[Appointment] = []

    def schedule_appointments(self, appointments: List[Appointment], sort_key=None):

        if sort_key is None:
            sort_key = lambda a: (a.priority, a.duration)

        for appt in sorted(appointments, key=sort_key):
            op_free, op_id = heapq.heappop(self._op_heap)
            dent_free, dent_id = heapq.heappop(self._dent_heap)

            start = max(op_free, dent_free)
            end = start + appt.duration

            if end > CLINIC_CLOSE_HOUR:

                heapq.heappush(self._op_heap, (op_free, op_id))
                heapq.heappush(self._dent_heap, (dent_free, dent_id))
                self.unscheduled.append(appt)
                continue

            appt.start, appt.end = start, end
            appt.operatory_id, appt.dentist_id = op_id, dent_id
            self.schedule.append(appt)

            heapq.heappush(self._op_heap, (end, op_id))
            heapq.heappush(self._dent_heap, (end, dent_id))

    def makespan(self) -> int:
        return max((a.end for a in self.schedule), default=0)

    def print_schedule(self):
        print(f"\n{'=' * 64}")
        print(f" CLINIC SCHEDULE  |  {self.num_operatories} operatories, "
              f"{self.num_dentists} dentists")
        print(f"{'=' * 64}")

        for op_id in range(1, self.num_operatories + 1):
            print(f"\n Operatory {op_id}:")
            appts = sorted(
                (a for a in self.schedule if a.operatory_id == op_id),
                key=lambda a: a.start,
            )
            if not appts:
                print("   (idle all day)")
            for a in appts:
                print(f"   {a.start:02d}:00 - {a.end:02d}:00   "
                      f"{a.patient:<15} (Dentist {a.dentist_id}, {a.duration}h)")

        if self.unscheduled:
            print(f"\n \u26a0 {len(self.unscheduled)} appointment(s) did NOT fit "
                  f"in the {CLINIC_CLOSE_HOUR - CLINIC_OPEN_HOUR}h day:")
            for a in self.unscheduled:
                print(f"   - {a.patient} ({a.duration}h, priority {a.priority})")

        span = self.makespan()
        hrs_used = span - CLINIC_OPEN_HOUR if span else 0
        print(f"\n All bookable operations finish by {span:02d}:00 "
              f"({hrs_used}h of clinic time used).")


def analyze_operatory_efficiency(appointments: List[Appointment], num_dentists: int,
                                  max_operatories: int = 10):
    print(f"\n{'=' * 64}")
    print(" EFFICIENCY ANALYSIS: effect of changing the number of operatories")
    print(f"{'=' * 64}")

    results = []
    for num_op in range(1, max_operatories + 1):

        trial_appts = [a.clone() for a in appointments]
        sched = ClinicScheduler(num_op, num_dentists)
        sched.schedule_appointments(trial_appts)

        span = sched.makespan()
        hrs_used = span - CLINIC_OPEN_HOUR if span else 0
        unscheduled = len(sched.unscheduled)
        results.append((num_op, hrs_used, unscheduled))

        flag = "  <-- all patients fit" if unscheduled == 0 else f"  ({unscheduled} unscheduled)"
        print(f"  {num_op:2d} operatories -> day finishes after {hrs_used:2d}h{flag}")

    fully_scheduled = [r for r in results if r[2] == 0]
    if fully_scheduled:
        best_hours = min(r[1] for r in fully_scheduled)
        best_count = min(r[0] for r in fully_scheduled if r[1] == best_hours)
        print(f"\n  Recommendation: {best_count} operatories is the sweet spot -- "
              f"every patient is seen and the day finishes in {best_hours}h. "
              f"Adding more rooms beyond {best_count} does not shorten the day further.")
    else:
        print("\n  Even with 10 operatories, not every patient fits in one clinic "
              "day with the current number of dentists -- consider adding dentists too.")
    return results


def sample_appointments() -> List[Appointment]:
    """A representative day's worth of patient requests."""
    return [
        Appointment("J. Smith",     duration=2, priority=1),
        Appointment("A. Rivera",    duration=1, priority=2),
        Appointment("M. Chen",      duration=3, priority=2),
        Appointment("K. Patel",     duration=1, priority=3),
        Appointment("D. Johnson",   duration=4, priority=1),
        Appointment("L. Nguyen",    duration=2, priority=3),
        Appointment("S. Brown",     duration=1, priority=4),
        Appointment("R. Garcia",    duration=2, priority=2),
        Appointment("T. Wilson",    duration=3, priority=3),
        Appointment("P. Davis",     duration=1, priority=1),
        Appointment("E. Martinez",  duration=2, priority=4),
        Appointment("N. Clark",     duration=1, priority=5),
    ]


def main():
    NUM_OPERATORIES = 3
    NUM_DENTISTS = 2

    appointments = sample_appointments()

    scheduler = ClinicScheduler(NUM_OPERATORIES, NUM_DENTISTS)
    scheduler.schedule_appointments(appointments)
    scheduler.print_schedule()

    analyze_operatory_efficiency(sample_appointments(), NUM_DENTISTS, max_operatories=10)


if __name__ == "__main__":
    main()



 CLINIC SCHEDULE  |  3 operatories, 2 dentists

 Operatory 1:
   09:00 - 10:00   P. Davis        (Dentist 1, 1h)
   11:00 - 12:00   A. Rivera       (Dentist 2, 1h)
   14:00 - 17:00   M. Chen         (Dentist 1, 3h)

 Operatory 2:
   09:00 - 11:00   J. Smith        (Dentist 2, 2h)
   12:00 - 14:00   R. Garcia       (Dentist 2, 2h)
   14:00 - 15:00   K. Patel        (Dentist 2, 1h)

 Operatory 3:
   10:00 - 14:00   D. Johnson      (Dentist 1, 4h)
   15:00 - 17:00   L. Nguyen       (Dentist 2, 2h)

 ⚠ 4 appointment(s) did NOT fit in the 8h day:
   - T. Wilson (3h, priority 3)
   - S. Brown (1h, priority 4)
   - E. Martinez (2h, priority 4)
   - N. Clark (1h, priority 5)

 All bookable operations finish by 17:00 (8h of clinic time used).

 EFFICIENCY ANALYSIS: effect of changing the number of operatories
   1 operatories -> day finishes after  8h  (8 unscheduled)
   2 operatories -> day finishes after  8h  (4 unscheduled)
   3 operatories -> day finishes after  8h  (4 unscheduled)
   4 op